In [ ]:
from config import get_config
from inference.infer_ann import run_inference
import numpy as np

config = get_config(
    model="ANN",
    target="H",
    stability="unstable",
    mode="predict"
)

# Example input (must match training features)
X_new = np.array([
    [0.1, 5.0, 300.0, 295.0],
    [0.2, 6.0, 302.0, 296.0]
])

y_pred = run_inference(X_new, config)

print(y_pred)

In [ ]:
import numpy as np
import pandas as pd
import joblib

from preprocessing.target_transform import TargetTransformer
from config import get_config

# -------------------------------
# BEST MODELS (from your table)
# -------------------------------
best_models = {
    ('ANN', 'H', 'stable'): 8,
    ('ANN', 'H', 'unstable'): 388,
    ('ANN', 'TAU', 'stable'): 124,
    ('ANN', 'TAU', 'unstable'): 392,

    ('RF', 'H', 'stable'): 372,
    ('RF', 'H', 'unstable'): 360,
    ('RF', 'TAU', 'stable'): 268,
    ('RF', 'TAU', 'unstable'): 384,

    ('XGB', 'H', 'stable'): 352,
    ('XGB', 'H', 'unstable'): 248,
    ('XGB', 'TAU', 'stable'): 4,
    ('XGB', 'TAU', 'unstable'): 188,
}


# -------------------------------
def load_model(model, target, stability, seed):
    path = f"saved_models/params/Ensemble2/{model}_{target}_{stability}_seed{seed}.pkl"
    print("[INFO] Loading:", path)
    return joblib.load(path)


def load_scaler(target, stability):
    path = f"saved_models/stats/Ensemble2/ANN_{target}_{stability}_scaler.pkl"
    print("[INFO] Loading scaler:", path)
    return joblib.load(path)


# -------------------------------
def predict_for_case(X, model_name, target, stability):

    seed = best_models[(model_name, target, stability)]

    config = get_config(
        model=model_name,
        target=target,
        stability=stability,
        mode="predict"
    )

    # -------- Load model --------
    model = load_model(model_name, target, stability, seed)

    X_input = X.copy()

    # -------- Scaling (ANN only) --------
    if model_name == "ANN":
        scaler = load_scaler(target, stability)
        X_input = scaler.transform(X_input)

    # -------- Predict --------
    y_pred = model.predict(X_input)

    # -------- Inverse transform --------
    transformer = TargetTransformer(config)
    y_pred = transformer.inverse_transform(y_pred)

    return y_pred


# -------------------------------
def run_full_inference(X_new):

    results = {}

    for target in ["H"]:
        for stability in ["stable"]:
            for model in ["ANN", "RF", "XGB"]:

                key = f"{model}_{target}"

                y_pred = predict_for_case(
                    X_new,
                    model,
                    target,
                    stability
                )

                results[key] = y_pred

    return results


In [ ]:
import numpy as np
import pandas as pd

no = 50000
g=9.81
# Bounds: [t27, u27, t0]
a = [275, 0.05, 265]
b = [305, 10, 310]
np.random.seed(42)
# Generate random samples
t27 = np.random.uniform(a[0], b[0], no)
u27 = np.random.uniform(a[1], b[1], no)
t0  = np.random.uniform(a[2], b[2], no)
Ri = (g * (t27-t0) * (27)) / (((t27+t0)/2.) * (u27**2))
# Create dataframe["u27","t27","t0","rib_0_27"]
df_rand = pd.DataFrame({
    'u27': u27,
    't27': t27,
    't0': t0,'rib_0_27': Ri
    
})
mask_unstable = df_rand['rib_0_27']<0
mask_stable = df_rand['rib_0_27']>0
mask_rib = (df_rand['rib_0_27'] >= -10) & (df_rand['rib_0_27'] <= 10)
mask_temp = (np.abs(df_rand['t0'] - df_rand['t27']) <= 10)


Y_stable

In [ ]:
import matplotlib.pyplot as plt

X_stable = df_rand[mask_stable&mask_rib &mask_temp].copy()
Y_stable= run_full_inference(X_stable)
for name, h in Y_stable.items():
    plt.figure()
    plt.plot(X_stable['rib_0_27'], h, '.', label=name)
    plt.xlabel('rib')
    plt.ylabel(name)
    plt.legend()
    plt.title(f"{name} vs rib")
    plt.show()
    x = np.log(X_stable['rib_0_27'].values)
    y = h
    
    # Remove invalid values
    mask = np.isfinite(x) & np.isfinite(y)& (x>=-4) & (x<=3)
    x = x[mask]
    y = y[mask]
    
    # ---- Bin in log space (Δlog = 0.01) ----
    bin_width = 0.5
    
    bins = np.arange(-4.25, 4 + bin_width, bin_width)
    
    # Create dataframe
    df_plot = pd.DataFrame({'x': x, 'y': y})
    df_plot['bin'] = pd.cut(df_plot['x'], bins)
    
    # Group
    grouped = df_plot.groupby('bin')
    
    # Prepare boxplot data
    box_data = []
    positions = []
    
    for interval, group in grouped:
        if len(group) > 5:   # avoid sparse bins
            box_data.append(group['y'].values)
            positions.append(interval.mid)  # midpoint in log space
    
    # ---- Plot ----
    plt.figure(figsize=(8,6))
    
    # Light scatter
    plt.scatter(x, y, alpha=0.1, s=5)
    
    # Boxplot
    plt.boxplot(box_data, positions=positions, widths=0.1,showfliers=False)
    
    plt.xlabel('log(-RiB)')
    plt.ylabel('Predicted Flux')
    plt.title(f'{name}_Stable')
    plt.grid(True)
    
    plt.show()

In [ ]:
X_unstable = df_rand[mask_unstable&mask_rib &mask_temp].copy()

Y_unstable=run_full_inference(X_unstable)
for name, h in Y_unstable.items():
    plt.figure()
    plt.plot(X_unstable['rib_0_27'], h, '.', label=name)
    plt.xlabel('rib')
    plt.ylabel(name)
    plt.legend()
    plt.title(f"{name} vs rib")
    plt.show()
    x = np.log(-X_unstable['rib_0_27'].values)
    y = h
    
    # Remove invalid values
    mask = np.isfinite(x) & np.isfinite(y)& (x>=-4) & (x<=3)
    x = x[mask]
    y = y[mask]
    
    # ---- Bin in log space (Δlog = 0.01) ----
    bin_width = 0.5
    
    bins = np.arange(-4.25, 4 + bin_width, bin_width)
    
    # Create dataframe
    df_plot = pd.DataFrame({'x': x, 'y': y})
    df_plot['bin'] = pd.cut(df_plot['x'], bins)
    
    # Group
    grouped = df_plot.groupby('bin')
    
    # Prepare boxplot data
    box_data = []
    positions = []
    
    for interval, group in grouped:
        if len(group) > 5:   # avoid sparse bins
            box_data.append(group['y'].values)
            positions.append(interval.mid)  # midpoint in log space
    
    # ---- Plot ----
    plt.figure(figsize=(8,6))
    
    # Light scatter
    plt.scatter(x, y, alpha=0.1, s=5)
    
    # Boxplot
    plt.boxplot(box_data, positions=positions, widths=0.1,showfliers=False)
    
    plt.xlabel('log(-RiB)')
    plt.ylabel('Predicted Flux')
    plt.title(f'{name}_Unstable')
    plt.grid(True)
    
    plt.show()

In [ ]:
x = np.log(-df_rand['rib'].values)
y = Y_pred_ran

# Remove invalid values
mask = np.isfinite(x) & np.isfinite(y)& (x>=-4) & (x<=3)
x = x[mask]
y = y[mask]

# ---- Bin in log space (Δlog = 0.01) ----
bin_width = 0.5

bins = np.arange(-4.25, 4 + bin_width, bin_width)

# Create dataframe
df_plot = pd.DataFrame({'x': x, 'y': y})
df_plot['bin'] = pd.cut(df_plot['x'], bins)

# Group
grouped = df_plot.groupby('bin')

# Prepare boxplot data
box_data = []
positions = []

for interval, group in grouped:
    if len(group) > 500:   # avoid sparse bins
        box_data.append(group['y'].values)
        positions.append(interval.mid)  # midpoint in log space

# ---- Plot ----
plt.figure(figsize=(8,6))

# Light scatter
plt.scatter(x, y, alpha=0.1, s=5)

# Boxplot
plt.boxplot(box_data, positions=positions, widths=0.1,showfliers=False)

plt.xlabel('log(-RiB)')
plt.ylabel('Predicted Flux')
plt.title('UNSATBLE-Boxplot in Log Space (Δlog = 0.5)')
plt.grid(True)

plt.show()